# Anomaly Detection: Unsupervised Learning

So far, we have used **Supervised Learning** (Logistic Regression, XGBoost). These models rely on the `isFraud` label being perfectly accurate. In the real world, many fraudulent transactions go undetected and are labeled as "Legitimate". 

**Unsupervised Anomaly Detection** helps us find "weird" or "unusual" transactions without relying on historical labels. This is critical for catching *zero-day fraud* (new patterns of fraud that haven't been seen before).

We will use **Isolation Forest**, an algorithm that isolates anomalies instead of profiling normal data points.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.preprocessing import load_and_clean_data, preprocess_and_split
from src.feature_engineering import engineer_features

### 1. Load Data
We use the same pipeline to ensure apples-to-apples comparison.

In [2]:
data_path = '../data/raw/PS_20174392719_1491204439457_log.csv'
df = load_and_clean_data(data_path)
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = preprocess_and_split(df, val_size=0.15, test_size=0.15, random_state=42)
print("Data ready!")

Data ready!


### 2. Train Isolation Forest
Because Isolation Forest is unsupervised, we only feed it `X_train`. We do not give it `y_train`.

We set `contamination=0.005` (0.5%) because we expect a very small percentage of anomalies. (The actual fraud rate is ~0.13%, but setting it slightly higher allows us to catch more potential fraud for human review).

In [3]:
# This might take a couple of minutes on 5M training rows
iso_forest = IsolationForest(n_estimators=100, contamination=0.005, random_state=42, n_jobs=-1)
print("Training Isolation Forest...")
iso_forest.fit(X_train)
print("Training complete.")

Training Isolation Forest...


Training complete.


### 3. Evaluate Unsupervised Model against Actual Labels
Isolation Forest outputs `-1` for anomalies (Fraud) and `1` for normal points. We need to convert these to `1` (Fraud) and `0` (Normal) to compare with our true labels.

In [4]:
from sklearn.metrics import average_precision_score
print("Predicting on validation set...")
y_pred_iso = iso_forest.predict(X_val)
y_prob_iso = -1 * iso_forest.decision_function(X_val)

y_pred_binary = np.where(y_pred_iso == -1, 1, 0)

print("--- Isolation Forest Evaluation ---")
print(classification_report(y_val, y_pred_binary))

pr_auc = average_precision_score(y_val, y_prob_iso)
print(f"PR-AUC: {pr_auc:.4f}")

import json
with open('../reports/anomaly_metrics.json', 'w') as f:
    json.dump({'pr_auc': pr_auc}, f, indent=4)
print("Saved anomaly metrics.")

Predicting on validation set...


--- Isolation Forest Evaluation ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    414330
           1       0.05      0.08      0.06      1232

    accuracy                           0.99    415562
   macro avg       0.52      0.54      0.53    415562
weighted avg       0.99      0.99      0.99    415562



PR-AUC: 0.0275
Saved anomaly metrics.


### Summary
Notice that the Precision for Fraud (Class 1) is likely much lower than XGBoost. This is expected! Isolation Forest isn't trying to predict the exact `isFraud` label; it's trying to find the weirdest transactions.

In a real-world pipeline, the Isolation Forest score is often used as a *Feature* inside the XGBoost model, combining the best of both supervised and unsupervised learning!